# 06 · Resolve customer, product and account mappings

Run this notebook from the `silver/` folder with the existing project kernel. It selects and verifies Bronze manifests, then rebuilds the required prior stages in memory. Reuse the same `silver_settings.json` to review a consistent selection across notebooks.

In [1]:
from IPython.display import display
import pandas as pd

from silver_notebook import build, frame, input_frame, settings_from_file

model = build("mdm", settings_from_file())
display(input_frame(model))

,source,manifest
0,erpnext,batch/erpnext/manual__2026-11-21T15%3A27%3A44....
1,dealer-sql,batch/dealer-sql/manual__2026-11-21T15%3A28%3A...
2,parent-pos,batch/parent-pos/manual__2026-11-21T15%3A27%3A...
3,dealer_export,files/otc-inbound/dealer/exports/dealer-export...
4,parent_finance,files/otc-inbound/parent/finance/dealer-order-...
5,dealer_finance,files/otc-inbound/dealer/finance/dealer-order-...
6,mdm,files/otc-inbound/seed-v1/8fab9f89f5242c6819f2...


The fixture supplies seeded decisions for Customer and Product and Finance-owned classifications for local accounts. A fixture alias must resolve to one local record before a crosswalk can be published. Approved Customer fixture pairs must also agree on the recorded registration ID. Product fixture approvals use the seeded decision file. Pending pairs retain their local record and do not receive a shared ID.

Identity approval and unit conversion are separate. Review the documented opt-in approval shape in the runbook if you are exercising the dealer seal pack case. Its original quantity and amount remain on the local order line.

In [2]:
display(frame(model, "match_candidate"))
display(frame(model, "customer_crosswalk"), frame(model, "product_crosswalk"))
display(frame(model, "account_crosswalk"), frame(model, "uom_conversion"))

,id,domain,parent_alias,dealer_alias,decision_status,reviewer,evidence
0,"[""customer"",""P-C1"",""D-C1""]",customer,P-C1,D-C1,approved,None,seed-v1 candidate manifest
1,"[""customer"",""P-C2"",""D-C2""]",customer,P-C2,D-C2,approved,None,seed-v1 candidate manifest
2,"[""customer"",""P-C3"",""D-C3""]",customer,P-C3,D-C3,pending,None,seed-v1 candidate manifest
3,"[""customer"",""P-C4"",""D-C4""]",customer,P-C4,D-C4,pending,None,seed-v1 candidate manifest
4,"[""customer"",""P-C5"",""D-C5""]",customer,P-C5,D-C5,separate,None,seed-v1 candidate manifest
5,"[""product"",""P-PUMP"",""D-PUMP""]",product,P-PUMP,D-PUMP,approved,None,seed-v1 candidate manifest
6,"[""product"",""P-VALVE"",""D-VALVE""]",product,P-VALVE,D-VALVE,approved,None,seed-v1 candidate manifest
7,"[""product"",""P-FILT"",""D-FLT""]",product,P-FILT,D-FLT,pending,None,seed-v1 candidate manifest
8,"[""product"",""P-SEAL"",""D-SEAL""]",product,P-SEAL,D-SEAL,pending,None,seed-v1 candidate manifest
9,"[""product"",""P-GAUGE"",""D-HOSE""]",product,P-GAUGE,D-HOSE,separate,None,seed-v1 candidate manifest


,id,source_id,group_id,valid_from,valid_to,match_candidate_id
0,"[""[\""parent\"",\""OTC P-C1 Metro Industrial Serv...","[""parent"",""OTC P-C1 Metro Industrial Services""]",G-CUST-001,2026-11-01,None,"[""customer"",""P-C1"",""D-C1""]"
1,"[""[\""dealer\"",\""D-C1\""]"",""2026-11-01""]","[""dealer"",""D-C1""]",G-CUST-001,2026-11-01,None,"[""customer"",""P-C1"",""D-C1""]"
2,"[""[\""parent\"",\""OTC P-C2 Pioneer Engineering\""...","[""parent"",""OTC P-C2 Pioneer Engineering""]",G-CUST-002,2026-11-01,None,"[""customer"",""P-C2"",""D-C2""]"
3,"[""[\""dealer\"",\""D-C2\""]"",""2026-11-01""]","[""dealer"",""D-C2""]",G-CUST-002,2026-11-01,None,"[""customer"",""P-C2"",""D-C2""]"
4,"[""[\""parent\"",\""OTC P-C5 Harbor Utilities\""]"",...","[""parent"",""OTC P-C5 Harbor Utilities""]",G-CUST-005,2026-11-01,None,"[""customer"",""P-C5"",""D-C5""]"
5,"[""[\""dealer\"",\""D-C5\""]"",""2026-11-01""]","[""dealer"",""D-C5""]",G-CUST-006,2026-11-01,None,"[""customer"",""P-C5"",""D-C5""]"


,id,source_id,group_id,valid_from,valid_to,match_candidate_id
0,"[""[\""parent\"",\""P-PUMP\""]"",""2026-11-01""]","[""parent"",""P-PUMP""]",G-PROD-001,2026-11-01,None,"[""product"",""P-PUMP"",""D-PUMP""]"
1,"[""[\""dealer\"",\""D-PUMP\""]"",""2026-11-01""]","[""dealer"",""D-PUMP""]",G-PROD-001,2026-11-01,None,"[""product"",""P-PUMP"",""D-PUMP""]"
2,"[""[\""parent\"",\""P-VALVE\""]"",""2026-11-01""]","[""parent"",""P-VALVE""]",G-PROD-002,2026-11-01,None,"[""product"",""P-VALVE"",""D-VALVE""]"
3,"[""[\""dealer\"",\""D-VALVE\""]"",""2026-11-01""]","[""dealer"",""D-VALVE""]",G-PROD-002,2026-11-01,None,"[""product"",""P-VALVE"",""D-VALVE""]"
4,"[""[\""parent\"",\""P-GAUGE\""]"",""2026-11-01""]","[""parent"",""P-GAUGE""]",G-PROD-005,2026-11-01,None,"[""product"",""P-GAUGE"",""D-HOSE""]"
5,"[""[\""dealer\"",\""D-HOSE\""]"",""2026-11-01""]","[""dealer"",""D-HOSE""]",G-PROD-006,2026-11-01,None,"[""product"",""P-GAUGE"",""D-HOSE""]"


,id,source_id,group_id,valid_from,valid_to
0,"[""[\""parent\"",\""Sales - OPD\""]"",""2026-11-01""]","[""parent"",""Sales - OPD""]",SALES_REVENUE,2026-11-01,None
1,"[""[\""parent\"",\""Debtors - OPD\""]"",""2026-11-01""]","[""parent"",""Debtors - OPD""]",TRADE_AR,2026-11-01,None
2,"[""[\""parent\"",\""45685245 - OTC Bank - OPD\""]"",...","[""parent"",""45685245 - OTC Bank - OPD""]",CASH_BANK,2026-11-01,None
3,"[""[\""parent\"",\""Stock In Hand - OPD\""]"",""2026-...","[""parent"",""Stock In Hand - OPD""]",INVENTORY,2026-11-01,None
4,"[""[\""parent\"",\""Cost of Goods Sold - OPD\""]"",""...","[""parent"",""Cost of Goods Sold - OPD""]",COGS,2026-11-01,None
5,"[""[\""dealer\"",\""400-D\""]"",""2026-11-01""]","[""dealer"",""400-D""]",SALES_REVENUE,2026-11-01,None
6,"[""[\""dealer\"",\""110-D\""]"",""2026-11-01""]","[""dealer"",""110-D""]",TRADE_AR,2026-11-01,None
7,"[""[\""dealer\"",\""100-D\""]"",""2026-11-01""]","[""dealer"",""100-D""]",CASH_BANK,2026-11-01,None
8,"[""[\""dealer\"",\""120-D\""]"",""2026-11-01""]","[""dealer"",""120-D""]",INVENTORY,2026-11-01,None
9,"[""[\""dealer\"",\""500-D\""]"",""2026-11-01""]","[""dealer"",""500-D""]",COGS,2026-11-01,None


""


In [3]:
conformed = frame(model, "order_line_conformed")
display(conformed)
if not conformed.empty:
    coverage = conformed.assign(
        customer_resolved=conformed["group_customer_id"].notna(),
        product_resolved=conformed["group_product_id"].notna(),
    )
    display(coverage.groupby(["legal_entity_id", "customer_resolved", "product_resolved"])
            .size().rename("lines").reset_index())
display(pd.DataFrame(model.exceptions))

,id,sales_order_id,legal_entity_id,group_customer_id,group_product_id,source_quantity,source_uom,conformed_quantity,conformed_uom,currency,amount
0,"[""parent"",""SAL-ORD-2026-00001"",""vqgdj6mnv1""]","[""parent"",""SAL-ORD-2026-00001""]",parent,G-CUST-001,G-PROD-001,10.0,Nos,10.0,each,INR,100000.0
1,"[""parent"",""SAL-ORD-2026-00002"",""vqk0j2tpqq""]","[""parent"",""SAL-ORD-2026-00002""]",parent,G-CUST-002,G-PROD-002,1.0,Nos,1.0,each,INR,10000.0
2,"[""parent"",""SAL-ORD-2026-00003"",""vqltad6jqq""]","[""parent"",""SAL-ORD-2026-00003""]",parent,None,None,1.0,Nos,None,None,INR,10000.0
3,"[""parent"",""SAL-ORD-2026-00004"",""vqlfquq375""]","[""parent"",""SAL-ORD-2026-00004""]",parent,None,None,1.0,Nos,None,None,INR,10000.0
4,"[""parent"",""SAL-ORD-2026-00005"",""vqm83vc86c""]","[""parent"",""SAL-ORD-2026-00005""]",parent,G-CUST-005,G-PROD-005,1.0,Nos,1.0,each,INR,10000.0
5,"[""parent"",""SAL-ORD-2026-00006"",""brbeshsujo""]","[""parent"",""SAL-ORD-2026-00006""]",parent,G-CUST-001,G-PROD-001,2.0,Nos,2.0,each,INR,20000.0
6,"[""dealer"",""D-BRONZE-R2-ORDER"",""D-BRONZE-R2-LINE""]","[""dealer"",""D-BRONZE-R2-ORDER""]",dealer,G-CUST-001,G-PROD-001,2,Each,2,each,INR,20000
7,"[""dealer"",""D-S1"",""D-L1""]","[""dealer"",""D-S1""]",dealer,G-CUST-001,G-PROD-001,1,Each,1,each,INR,40000
8,"[""dealer"",""D-S2"",""D-L2""]","[""dealer"",""D-S2""]",dealer,G-CUST-002,G-PROD-002,1,Each,1,each,INR,10000
9,"[""dealer"",""D-S3"",""D-L3""]","[""dealer"",""D-S3""]",dealer,None,None,1,Each,None,None,INR,10000


,legal_entity_id,customer_resolved,product_resolved,lines
0,dealer,False,False,2
1,dealer,True,True,4
2,parent,False,False,2
3,parent,True,True,4


""
